## Version 1

## INIT

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("LFM_Training") \
    .getOrCreate()

bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
26/03/18 07:02:35 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:02:35 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 07:02:35 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Pleas

In [3]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

date = "2026-03-17"
required_days = 30

In [4]:
daily_watch_history_path = "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new"
date_format = "%Y-%m-%d"
base_date = datetime.strptime(date, date_format)

paths = [
    f"{daily_watch_history_path}/day={(base_date - timedelta(days=i)).strftime(date_format)}"
    for i in range(required_days)
]

watch_history_df = None

for path in paths:
    try:
        temp_df = spark.read.parquet(path)

        if watch_history_df is None:
            watch_history_df = temp_df
        else:
            watch_history_df = watch_history_df.unionByName(temp_df)

    except Exception:
        print(f"Skipping missing path: {path}")

In [5]:
enriched_db_path = "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/" 

enriched_tv_df = spark.read.parquet(f'{enriched_db_path}{date}/enriched_tv.parquet')
enriched_movies_df = spark.read.parquet(f'{enriched_db_path}{date}/enriched_movie.parquet')

26/03/18 07:02:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Cleaning DB

In [8]:
# 1. Ignore files that are physically broken/incomplete
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

# 2. Disable the Vectorized Reader (Standardizes the reading process)
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# 3. Handle schema mismatches if they exist
spark.conf.set("spark.sql.parquet.mergeSchema", "true")


filtered_enriched_tv = (
    enriched_tv_df
    # 1. Remove empty arrays and un-published content
    .filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
    # 2. Explode and select in one go
    .select(
        F.explode("XstreamContentIds").alias("item_id"), # This is your primary key for ALS joins
        "title", 
        "ID",
        F.col('OriginalLanguage').alias('original_language'), 
        "Genres"
    )
)
filtered_enriched_movies = (
    enriched_movies_df
    # 1. Remove empty arrays and un-published content
    .filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
    # 2. Explode and select in one go
    .select(
        F.explode("XstreamContentIds").alias("item_id"), # This is your primary key for ALS joins
        "title", 
        "ID", 
        F.col('OriginalLanguage').alias('original_language'),
        "Genres"
    )
)


# 4. Perform the union with explicit casting to ensure types match
tv_final = filtered_enriched_tv.select(
    F.col("item_id").cast("string"),
    F.col("title").cast("string"),
    F.col("original_language").cast("string"),
    F.col("Genres").cast("string") # Ensure both are cast to the same type
)

movies_final = filtered_enriched_movies.select(
    F.col("item_id").cast("string"),
    F.col("title").cast("string"),
    F.col("original_language").cast("string"),
    F.col("Genres").cast("string")
)

enriched_content_df = tv_final.unionByName(movies_final).distinct()
# enriched_content_df.cache().count() # Use count() to force Spark to read and verify all files now
enriched_content_df = filtered_enriched_tv.unionByName(filtered_enriched_movies)


26/03/18 07:03:58 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 2 for reason Executor for container container_1764236692086_5266_01_000002 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.


In [7]:
filtered_enriched_tv.show(5, truncate=False)
# filtered_enriched_movies.show(5, truncate=False)

+---------------------------------------------------------------------+--------------------------------------------+----------+-----------------+-------------------------------------------------+
|item_id                                                              |title                                       |ID        |original_language|Genres                                           |
+---------------------------------------------------------------------+--------------------------------------------+----------+-----------------+-------------------------------------------------+
|MINITV_TVSHOW_amzn1.dv.gti.510882ec-b063-435b-a448-7c2f6a998357      |JoJo no Kimyô na Bôken                      |nftv_45790|ja               |[Animation, Action & Adventure, Sci-Fi & Fantasy]|
|MINITV_TVSHOW_amzn1.dv.gti.6c59b17c-8235-4261-89dd-24ee9b70a048      |Yôkoso jitsuryoku shijô shugi no kyôshitsu e|nftv_72517|en               |[Animation, Drama, Mystery]                      |
|HOTSTAR_DTH_TVSHOW_

Combining playtimes

In [9]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# 1. Aggregate Playtime and Count Distinct items per user in one flow
user_stats = (
    watch_history_df  
    .groupBy("userId", "item_id")
    .agg(
        F.sum("total_play_time_sec").alias("total_playtime_combined")
    )
    # We add a column to count how many distinct items EACH user has watched
    .withColumn("distinct_content_count", F.count("item_id").over(Window.partitionBy("userId")))
)

# 2. Filter for users with at least 3 distinct items
als_input_base = user_stats.filter("distinct_content_count >= 3")

## ALS

ALS data

In [12]:
import pyspark.sql.functions as F
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.recommendation import ALS

# 1. Clean data FIRST to avoid indexing useless rows
clean_base = als_input_base.filter(
    F.col("total_playtime_combined").isNotNull() & 
    ~F.isnan("total_playtime_combined")
)

# 2. Set up Indexers
# stringOrderType="alphabetAsc" is generally faster on massive datasets than the 
# default frequency count, which requires a heavy aggregation step.
user_indexer = StringIndexer(
    inputCol="userId", 
    outputCol="userIndex", 
    stringOrderType="alphabetAsc",
    handleInvalid="skip"
)
item_indexer = StringIndexer(
    inputCol="item_id",
    outputCol="itemIndex", 
    stringOrderType="alphabetAsc",
    handleInvalid="skip"
)

# 3. Run the Pipeline (This replaces distinct, row_number, and joins!)
pipeline = Pipeline(stages=[user_indexer, item_indexer])
indexed_data = pipeline.fit(clean_base).transform(clean_base)

item_lookup = indexed_data.select("item_id", "itemIndex").distinct()

# 4. Transform, Repartition, and Cache
als_data = (
    indexed_data
    .withColumn("playtime_logged", F.log1p("total_playtime_combined"))
    # The .cast("int") strips the heavy StringIndexer metadata out of the schema!
    .select(
        F.col("userIndex").cast("int").alias("userIndex"), 
        F.col("itemIndex").cast("int").alias("itemIndex"), 
        "playtime_logged"
    )
    .repartition(1000) 
)

als_data.cache()
als_data.show(5, truncate=False)

# 5. Train the Model
als = ALS(
    userCol="userIndex",
    itemCol="itemIndex",
    ratingCol="playtime_logged",
    implicitPrefs=True,
    rank=20,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop"
)

model = als.fit(als_data)

26/03/18 07:05:53 WARN DAGScheduler: Broadcasting large task binary with size 77.8 MiB
26/03/18 07:06:00 WARN DAGScheduler: Broadcasting large task binary with size 77.8 MiB
26/03/18 07:06:37 WARN DAGScheduler: Broadcasting large task binary with size 77.9 MiB
26/03/18 07:06:47 WARN DAGScheduler: Broadcasting large task binary with size 77.8 MiB
26/03/18 07:06:56 WARN DAGScheduler: Broadcasting large task binary with size 77.8 MiB
26/03/18 07:07:05 WARN DAGScheduler: Broadcasting large task binary with size 77.8 MiB
26/03/18 07:07:41 WARN DAGScheduler: Broadcasting large task binary with size 77.9 MiB
26/03/18 07:07:50 WARN DAGScheduler: Broadcasting large task binary with size 77.8 MiB
26/03/18 07:08:05 WARN DAGScheduler: Broadcasting large task binary with size 89.7 MiB


+---------+---------+------------------+
|userIndex|itemIndex|playtime_logged   |
+---------+---------+------------------+
|854301   |23634    |7.703459047867175 |
|2091691  |19845    |8.882331182532061 |
|2007077  |12862    |6.7671854603628265|
|1698450  |10737    |7.81762544305337  |
|1491138  |2357     |2.1144466508785262|
+---------+---------+------------------+
only showing top 5 rows



26/03/18 07:08:19 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 73 for reason Container marked as failed: container_1764236692086_5266_01_000101 on host: dprc-wynk-prd-data-ds-machine-learning-common-sw-3t14.c.prj-wynk-prd-ds-svc-01.internal. Exit status: -100. Diagnostics: Container released on a *lost* node.
26/03/18 07:08:19 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 67 for reason Executor for container container_1764236692086_5266_01_000090 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 07:08:19 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 68 for reason Container marked as failed: container_1764236692086_5266_01_000093 on host: dprc-wynk-prd-data-ds-machine-learning-common-sw-dcsr.c.prj-wynk-prd-ds-svc-01.internal. Exit status: -100. Diagnostics: Container released on a *lost* node.
26/03/18 07:08:19 

Training

## Saving and loading Trained ALS model

In [11]:

# als_model_path = "gs://wynk-ml-workspace/_temp/harshith/als/als_model"
# user_lookup_path = "gs://wynk-ml-workspace/_temp/harshith/als/user_lookup"
# item_lookup_path = "gs://wynk-ml-workspace/_temp/harshith/als/item_lookup"

# # 1. Save the ALS Model (Uses parentheses for write and overwrite)
# model.write().overwrite().save(als_model_path)

# # 2. Save the Lookup DataFrames
# # Use .mode("overwrite") instead of .overwrite
# user_lookup.write.mode("overwrite").save(user_lookup_path)
# item_lookup.write.mode("overwrite").save(item_lookup_path)

# print("✅ Models and Lookups successfully saved to GCS!")

# print("✅ Models successfully saved!")

In [12]:
# from pyspark.ml.recommendation import ALSModel

# # Load the Model
# model = ALSModel.load(als_model_path)

# # Read the Lookups
# user_lookup = spark.read.parquet(user_lookup_path)
# item_lookup = spark.read.parquet(item_lookup_path)

# print("✅ Model and Lookups loaded. Ready to recommend.")

## 

### Generating recommendations

In [14]:
user_recommendations = model.recommendForAllUsers(15)
# user_recommendations.show(10, truncate=False)

from pyspark.sql.functions import col, explode

# 1. Explode ALL recommendations for ALL users
all_recs_exploded = user_recommendations.select(
    col("userIndex"),
    explode("recommendations").alias("rec")
).select(
    "userIndex",
    col("rec.itemIndex").alias("itemIndex"),
    col("rec.rating").alias("score")
)

# 2. Perform a Global Anti-Join 
# This removes EVERY user-item pair that already exists in your training data
clean_recommendations = all_recs_exploded.join(
    als_data, # Your original training data
    on=["userIndex", "itemIndex"], 
    how="left_anti"
)
item_lookup = indexed_data.select("item_id", "itemIndex").distinct()
# 3. Cache this "Clean" version
clean_recommendations.cache().show(10, truncate=False)

+---------+---------+----------+
|userIndex|itemIndex|score     |
+---------+---------+----------+
|30       |9916     |0.15185788|
|57       |578      |0.26766372|
|116      |844      |0.6052091 |
|137      |6694     |0.46229008|
|142      |27084    |0.4444842 |
|177      |21123    |0.44679546|
|258      |19816    |0.29309514|
|270      |13212    |0.27709183|
|337      |26813    |0.273558  |
|386      |135      |0.0       |
+---------+---------+----------+
only showing top 10 rows



## Viewing Results

### Show recommendations by User

In [ ]:
# Pick a random user index from the CLEANED recommendations

random_user_idx = clean_recommendations.select("userIndex").sample(False, 0.1).limit(1).first()[0]

print(f"--- Recommendations for User Index: {random_user_idx} ---")


history = als_data.filter(col("userIndex") == random_user_idx).join(item_lookup, "itemIndex")

reco_10 = (clean_recommendations
    .filter(col("userIndex") == random_user_idx)
    .join(item_lookup, "itemIndex")
    .orderBy(col("score").desc())
    .limit(10))

# Show History
print("\n[ History ]")
history = history.join(enriched_content_df,"item_id", how="left").select("item_id", "title", "original_language", "Genres")
history.show(10, truncate=False)

# Show Clean Recs
print("\n[ New Suggestions Only ]")
reco_10 = reco_10.join(enriched_content_df,"item_id", how = "left").select("item_id", "title", "original_language", "Genres")
reco_10.show(10, truncate=False)

--- Recommendations for User Index: 38 ---

[ History ]


26/03/18 06:10:14 WARN DAGScheduler: Broadcasting large task binary with size 1544.1 KiB
26/03/18 06:10:18 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 06:10:46 WARN DAGScheduler: Broadcasting large task binary with size 77.1 MiB
26/03/18 06:10:57 WARN DAGScheduler: Broadcasting large task binary with size 77.0 MiB
26/03/18 06:11:06 WARN DAGScheduler: Broadcasting large task binary with size 78.6 MiB
26/03/18 06:11:15 WARN DAGScheduler: Broadcasting large task binary with size 78.6 MiB
26/03/18 06:11:16 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 06:11:17 WARN SparkConf: The configuration key 'spark.yarn.e

+---------------------------------------------------------------+-------------------------------+-----------------+----------------------------------------------+
|item_id                                                        |title                          |original_language|Genres                                        |
+---------------------------------------------------------------+-------------------------------+-----------------+----------------------------------------------+
|ZEEFIVE_MOVIE_0-0-2539                                         |Tamasha                        |hi               |[Romance, Drama, Comedy]                      |
|EROSNOW_MOVIE_1000637                                          |Rockstar                       |hi               |[Drama, Music, Romance]                       |
|MINITV_MOVIE_amzn1.dv.gti.333f80ac-c7b8-4963-9b70-3d48c9031ec4 |Rocky Aur Rani Kii Prem Kahaani|hi               |[Comedy, Drama, Family, Romance]              |
|MINITV_TVSHOW_amzn1.d

26/03/18 06:11:17 WARN DAGScheduler: Broadcasting large task binary with size 1544.1 KiB
26/03/18 06:11:18 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 06:11:52 WARN DAGScheduler: Broadcasting large task binary with size 77.1 MiB
26/03/18 06:11:56 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 06:11:58 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 06:11:58 WARN SparkConf: The configuration key 'spark.yarn.executor.failur

+---------------------------------------------------------------+-------------------------------+-----------------+--------------------------------------------+
|item_id                                                        |title                          |original_language|Genres                                      |
+---------------------------------------------------------------+-------------------------------+-----------------+--------------------------------------------+
|SONYLIV_VOD_TVSHOW_1700001014                                  |Indian Idol                    |hi               |[Reality]                                   |
|SONYLIV_VOD_TVSHOW_1790006756                                  |Wheel Of Fortune               |hi               |[Drama, Reality, Comedy, Family]            |
|MINITV_TVSHOW_amzn1.dv.gti.f1c6827c-5064-419d-97ca-c53be67e7c8a|Bhay: The Gaurav Tiwari Mystery|hi               |[Drama, Mystery]                            |
|MINITV_MOVIE_amzn1.dv.gti.ac3855c

In [16]:
item_factors = model.itemFactors
item_features_final = (
    model.itemFactors
    .join(item_lookup, col("id") == col("itemIndex"))
    .select(
        "item_id",      # Your actual content ID/Name
        "itemIndex",    # The integer used by ALS
        "features"      # The latent vector [rank_1, rank_2, ...]
    )
)

### Show Recommendations by ID

In [17]:
# # --- CONFIGURATION ---
# TARGET_ITEM_ID = "SHEMAROOME_MOVIE_5b9fb803c1df41453400012b"  # Replace with the actual ID you want to look up
# # ---------------------

# # 1. Get the vector for your target item
# target_row = item_features_final.filter(col("item_id") == TARGET_ITEM_ID).select("features").first()
# enriched_content_df.filter(col("item_id") == TARGET_ITEM_ID).select("title", "ID", "item_id", "original_language", "Genres").show(truncate=False)
# if not target_row:
#     print(f"Item {TARGET_ITEM_ID} not found in the dataset.")
# else:
#     target_vector = target_row[0]
#     num_features = len(target_vector)

#     # 2. Calculate Similarity (Dot Product)
#     # This creates a column where we multiply target[i] * features[i] for every dimension
#     similar_items = (
#         item_features_final
#         .withColumn("similarity_score", 
#             sum(col("features")[i] * target_vector[i] for i in range(num_features))
#         )
#         # 3. Filter out the item itself so you don't recommend the same thing
#         .filter(col("item_id") != TARGET_ITEM_ID)
#         .select("item_id", "similarity_score")
#         .orderBy(col("similarity_score").desc())
#         .limit(10)
#         .join(enriched_content_df, "item_id", how = "left")
#     )

#     print(f"✨ Items most similar to: {TARGET_ITEM_ID}")
#     similar_items.select("title", "ID", "item_id", "original_language", "Genres", "similarity_score").show(truncate=False)
# # item_features_final.show(10, truncate=False)

## LightFM

In [15]:
import pandas as pd
import numpy as np
import ast
from lightfm.data import Dataset

# =====================================================================
# 1. FETCH DATA FROM SPARK TO PANDAS
# =====================================================================
print("Fetching interactions from Spark to Pandas...")
# WARNING: Ensure als_data is filtered/sampled down from the 100GB size!
interactions_pdf = als_data.select("userIndex", "itemIndex", "playtime_logged").toPandas()
print(f" -> Interactions loaded: {len(interactions_pdf)} rows")

print("Fetching items metadata from Spark to Pandas...")
items_pdf = (
    enriched_content_df.join(item_lookup, "item_id")
    .select("itemIndex", "item_id", "title", "original_language", "Genres")
    .dropDuplicates(["itemIndex"])
    .toPandas()
)
print(f" -> Items loaded: {len(items_pdf)} rows")

# =====================================================================
# 2. CLEAN DATA AND CAST TYPES
# =====================================================================
print("Cleaning missing values and casting integer types...")
items_pdf['Genres'] = items_pdf['Genres'].fillna('Unknown').astype(str)
items_pdf['original_language'] = items_pdf['original_language'].fillna('Unknown').astype(str)

interactions_pdf['userIndex'] = interactions_pdf['userIndex'].astype(int)
interactions_pdf['itemIndex'] = interactions_pdf['itemIndex'].astype(int)
items_pdf['itemIndex'] = items_pdf['itemIndex'].astype(int)

# =====================================================================
# 3. FILTER INTERACTIONS TO MATCH VALID ITEMS
# =====================================================================
print("Filtering orphan interactions...")
valid_item_indices = items_pdf['itemIndex'].unique()
interactions_pdf = interactions_pdf[interactions_pdf['itemIndex'].isin(valid_item_indices)]

valid_user_indices = interactions_pdf['userIndex'].unique()
print(f" -> Filtered to {len(interactions_pdf)} valid interactions.")

# =====================================================================
# 4. THE UNIVERSAL FEATURE FLATTENER
# =====================================================================
def extract_flat_features(lang, genre_data):
    # Prefix the language
    features = [f"lang:{str(lang).strip()}"]
    
    if isinstance(genre_data, str):
        if genre_data.startswith('['):
            try:
                genre_data = ast.literal_eval(genre_data)
            except:
                genre_data = genre_data.replace('[', '').replace(']', '').replace("'", "").replace('"', "")
                genre_data = [g.strip() for g in genre_data.split(',')]
        else:
            genre_data = [g.strip() for g in genre_data.split(',')]
            
    if isinstance(genre_data, list):
        for item in genre_data:
            if isinstance(item, list):
                # Prefix the genre
                features.extend([f"genre:{str(i).strip()}" for i in item])
            else:
                features.append(f"genre:{str(item).strip()}")
                
    return features

# Apply this logic ONCE to create a perfect list of features for every item
print("Processing item features...")
items_pdf['clean_features'] = items_pdf.apply(
    lambda row: extract_flat_features(row['original_language'], row['Genres']), axis=1
)
print(f" -> Sample clean features: {items_pdf['clean_features'].iloc[0]}")

# =====================================================================
# 5. EXTRACT UNIQUE FEATURES FOR LIGHTFM DATASET
# =====================================================================
print("Extracting global unique features...")
all_item_features = set()
for feature_list in items_pdf['clean_features']:
    all_item_features.update(feature_list)
    
all_item_features = list(all_item_features)
print(f" -> Total unique item features found: {len(all_item_features)}")

# =====================================================================
# 6. INITIALIZE AND FIT LIGHTFM DATASET
# =====================================================================
print("Fitting LightFM Dataset mapping...")
dataset = Dataset()
dataset.fit(
    users=valid_user_indices,
    items=valid_item_indices,
    item_features=all_item_features
)
num_users, num_items = dataset.interactions_shape()
print(f" -> Dataset successfully mapped: {num_users} users and {num_items} items.")

# =====================================================================
# 7. BUILD LIGHTFM MATRICES
# =====================================================================
print("Building Interactions matrix...")
(interactions, weights) = dataset.build_interactions(
    zip(
        interactions_pdf['userIndex'], 
        interactions_pdf['itemIndex'], 
        interactions_pdf['playtime_logged']
    )
)

print("Building Item Features matrix...")
item_features = dataset.build_item_features(
    ((item_idx, features) 
    for item_idx, features in zip(
        items_pdf['itemIndex'], 
        items_pdf['clean_features'] 
    )),
    normalize=False # <--- CRITICAL FIX
)

print("\nMatrices built successfully! Ready to train the model.")

Fetching interactions from Spark to Pandas...
 -> Interactions loaded: 18625840 rows
Fetching items metadata from Spark to Pandas...


26/03/18 07:10:58 WARN DAGScheduler: Broadcasting large task binary with size 1549.1 KiB
26/03/18 07:10:59 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:11:36 WARN DAGScheduler: Broadcasting large task binary with size 77.9 MiB
26/03/18 07:11:43 WARN DAGScheduler: Broadcasting large task binary with size 77.8 MiB
26/03/18 07:11:54 WARN DAGScheduler: Broadcasting large task binary with size 79.4 MiB
26/03/18 07:12:01 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 07:12:04 WARN DAGScheduler: Broadcasting large task binary with size 79.5 MiB
26/03/18 07:12:10 WARN DAGScheduler: Broadcasting large task binary w

 -> Items loaded: 14691 rows
Cleaning missing values and casting integer types...
Filtering orphan interactions...
 -> Filtered to 13771670 valid interactions.
Processing item features...
 -> Sample clean features: ['lang:te', 'genre:Romance']
Extracting global unique features...
 -> Total unique item features found: 88
Fitting LightFM Dataset mapping...
 -> Dataset successfully mapped: 2138512 users and 14691 items.
Building Interactions matrix...
Building Item Features matrix...

Matrices built successfully! Ready to train the model.


In [16]:
from lightfm import LightFM

# Initialize the model (WARP is best for implicit ranking)
lfm_model = LightFM(
    loss='warp',
    no_components=30,
    learning_rate=0.03,
    item_alpha=1e-6,
    user_alpha=1e-6
)

print("Training LightFM model...")
lfm_model.fit(
    interactions,
    item_features=item_features,
    # sample_weight=weights,
    epochs=50,
    num_threads=4 
)
print("Training complete!")

Training LightFM model...


26/03/18 07:13:38 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 77 for reason Executor for container container_1764236692086_5266_01_000110 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 07:13:38 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 74 for reason Executor for container container_1764236692086_5266_01_000105 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 07:13:38 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 76 for reason Executor for container container_1764236692086_5266_01_000108 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 07:13:38 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 75 for reason Executor for container container_1764236692086_5

Training complete!


In [17]:
# Get LightFM's internal mappings
user_id_map, user_feature_map, item_id_map, item_feature_map = dataset.mapping()

# Create a reverse dictionary to map LightFM's internal item index back to your Spark itemIndex
inverse_item_id_map = {v: k for k, v in item_id_map.items()}

def recommend_for_user(target_user_index, model, interactions, item_features, num_recs=10):
    # Convert Spark userIndex to LightFM internal user index
    internal_user_id = user_id_map.get(target_user_index)
    if internal_user_id is None:
        return "User not found."
    
    # Get internal indices of items the user has already watched
    known_positives = interactions.tocsr()[internal_user_id].indices
    
    # Predict scores for all items
    n_items = len(item_id_map)
    scores = model.predict(
        internal_user_id, 
        np.arange(n_items), 
        item_features=item_features
    )
    
    # Rank the items
    top_items_indices = np.argsort(-scores)
    
    # Filter out known positives
    top_items_indices = [idx for idx in top_items_indices if idx not in known_positives]
    
    # Get top N Spark itemIndexes
    top_spark_item_indexes = [inverse_item_id_map[idx] for idx in top_items_indices[:num_recs]]
    
    # Return the readable dataframe
    return items_pdf[items_pdf['itemIndex'].isin(top_spark_item_indexes)][['item_id', 'title', 'original_language', 'Genres']]


In [21]:
# print(f"--- LightFM Recommendations for User Index: {random_user_idx} ---")
# print(recommend_for_user(random_user_idx, lfm_model, interactions, item_features))

## ALS vs LighFM comparisons

### Random User

In [19]:
# item_lookup = indexed_data.select("item_id", "itemIndex").distinct()

In [25]:
# Pick a random user index from the CLEANED recommendations

# 1. Define the item_lookup correctly (Overwriting the old Window version)
random_user_idx = (
    clean_recommendations
    .select("userIndex")
    .distinct() # Optional: Ensures you don't pick a user with heavy weighting more often
    .orderBy(F.rand()) 
    .limit(1)
    .first()[0]
)

print(f"--- Recommendations for User Index: {random_user_idx} ---")


history = als_data.filter(col("userIndex") == random_user_idx).join(item_lookup, "itemIndex")

reco_10 = (clean_recommendations
    .filter(col("userIndex") == random_user_idx)
    .join(item_lookup, "itemIndex")
    .orderBy(col("score").desc())
    .limit(10))

# Show History
print("\n[ History ]")
history = history.join(enriched_content_df,"item_id", how="left").select("item_id", "title", "original_language", "Genres")
history.show(10, truncate=False)

# Show Clean Recs
print("\n[ ALS Suggestions ]")
reco_10 = reco_10.join(enriched_content_df,"item_id", how = "left").select("item_id", "title", "original_language", "Genres")
reco_10.show(10, truncate=False)

print(f"--- LightFM Recommendations for User Index: {random_user_idx} ---")
# Get the Pandas DataFrame from your function
lightfm_pandas_df = recommend_for_user(random_user_idx, lfm_model, interactions, item_features)

# Convert it to Spark and print using the exact same formatting!
lightfm_spark_df = spark.createDataFrame(lightfm_pandas_df)
lightfm_spark_df.select("item_id", "title", "original_language", "Genres").show(10, truncate=False)

26/03/18 09:33:51 WARN TaskSetManager: Lost task 196.0 in stage 1368.0 (TID 61113) (dprc-wynk-prd-data-ds-machine-learning-common-sw-bz5l.c.prj-wynk-prd-ds-svc-01.internal executor 15): FetchFailed(BlockManagerId(35, dprc-wynk-prd-data-ds-machine-learning-common-sw-7hr7.c.prj-wynk-prd-ds-svc-01.internal, 43501, None), shuffleId=43, mapIndex=29, mapId=14530, reduceId=392, message=
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:437)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1382)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:1083)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:88)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.c

--- Recommendations for User Index: 1021368 ---

[ History ]


26/03/18 09:34:25 WARN DAGScheduler: Broadcasting large task binary with size 1549.1 KiB
26/03/18 09:34:26 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 09:34:57 WARN DAGScheduler: Broadcasting large task binary with size 77.9 MiB
26/03/18 09:35:04 WARN DAGScheduler: Broadcasting large task binary with size 77.8 MiB
26/03/18 09:35:14 WARN DAGScheduler: Broadcasting large task binary with size 79.4 MiB
26/03/18 09:35:23 WARN DAGScheduler: Broadcasting large task binary with size 79.4 MiB
26/03/18 09:35:24 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 09:35:25 WARN SparkConf: The configuration key 'spark.yarn.e

+----------------------------------------------+-----------+-----------------+-----------------------+
|item_id                                       |title      |original_language|Genres                 |
+----------------------------------------------+-----------+-----------------+-----------------------+
|ZEEFIVE_TVSHOW_0-6-4z5852811                  |NULL       |NULL             |NULL                   |
|AHA_MOVIE_CE1D0BE3-EA7A-47C7-BAFC-076C46071884|Eleven     |te               |[Thriller]             |
|AHA_MOVIE_DB256170-563E-4A94-92B3-BC98CA1D6AF2|Junior     |te               |[Drama, Family]        |
|ZEEFIVE_TVSHOW_0-6-4z5910206                  |Raakshasa  |kn               |[]                     |
|SONYLIV_VOD_TVSHOW_1700001072                 |Meet Cute  |te               |[Drama, Comedy, Family]|
|AHA_MOVIE_841C0A2D-DFA5-4DDF-B45A-BA04C037444A|Baby       |te               |[Romance, Drama]       |
|ZEEFIVE_MOVIE_0-0-1z5914226                   |Pathirathri|ml           

26/03/18 09:35:25 WARN DAGScheduler: Broadcasting large task binary with size 1549.1 KiB
26/03/18 09:35:27 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 09:35:59 WARN DAGScheduler: Broadcasting large task binary with size 77.9 MiB
26/03/18 09:36:06 WARN DAGScheduler: Broadcasting large task binary with size 77.8 MiB
26/03/18 09:36:15 WARN DAGScheduler: Broadcasting large task binary with size 79.4 MiB
26/03/18 09:36:24 WARN DAGScheduler: Broadcasting large task binary with size 79.4 MiB
26/03/18 09:36:25 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/18 09:36:25 WARN SparkConf: The configuration key 'spark.yarn.e

+----------------------------------------------+------------+-----------------+--------------------------+
|item_id                                       |title       |original_language|Genres                    |
+----------------------------------------------+------------+-----------------+--------------------------+
|ZEEFIVE_TVSHOW_0-6-4z5765779                  |NULL        |NULL             |NULL                      |
|ZEEFIVE_TVSHOW_0-6-4z5364077                  |Amruthadhare|kn               |[Drama]                   |
|ZEEFIVE_MOVIE_0-0-1z5895203                   |45          |kn               |[Action]                  |
|ZEEFIVE_TVSHOW_0-6-4z5735048                  |Ayyana Mane |kn               |[Crime, Mystery]          |
|ZEEFIVE_TVSHOW_0-6-4z5598544                  |NULL        |NULL             |NULL                      |
|AHA_MOVIE_91B37D19-AF75-4184-884F-7C2EC2809368|Shambhala   |te               |[Action, Horror, Thriller]|
|ZEEFIVE_TVSHOW_0-6-4z5478153        

+-----------------------------------------------+------------------------------+-----------------+--------------------------------+
|item_id                                        |title                         |original_language|Genres                          |
+-----------------------------------------------+------------------------------+-----------------+--------------------------------+
|AHA_TVSHOW_1D67B609-AC68-4A94-B80E-D0B8437B5519|3 Roses                       |te               |['Family', 'Comedy']            |
|SONYLIV_VOD_MOVIE_1090503578                   |Baby Girl                     |ml               |['Drama', 'Thriller']           |
|ZEEFIVE_MOVIE_0-0-1z5904059                    |Mana ShankaraVaraprasad Garu  |te               |['Action', 'Comedy', 'Mystery'] |
|AHA_MOVIE_0DC150B0-1DDA-41F5-8971-51688F2EBA22 |Om Shanti Shanti Shantihi     |te               |[]                              |
|AHA_MOVIE_0F598AED-27BE-451B-8B5D-3609D0A43767 |Psych Siddhartha           

26/03/18 09:38:04 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 101 for reason Executor for container container_1764236692086_5266_01_000195 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 09:38:04 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 102 for reason Executor for container container_1764236692086_5266_01_000196 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 09:38:04 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 103 for reason Executor for container container_1764236692086_5266_01_000197 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 09:38:04 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 104 for reason Executor for container container_17642366920

### Particular user

In [ ]:
# 1. Create a distinct mapping from your indexed data
user_mapping = indexed_data.select("userId", "userIndex").distinct()


In [20]:
# Pick a random user index from the CLEANED recommendations


# random_user_idx = (
#     clean_recommendations
#     .select("userIndex")
#     .distinct() # Optional: Ensures you don't pick a user with heavy weighting more often
#     .orderBy(F.rand()) 
#     .limit(1)
#     .first()[0]
# )

from pyspark.sql import functions as F


# 2. Filter for your specific ID
target_id = "ib-xPv06AjuYOE4BI0"
result = user_mapping.filter(F.col("userId") == target_id).collect()

if result:
    print(f"UserIndex: {int(result[0]['userIndex'])}")
    target_user_idx = int(result[0]['userIndex'])
    # target_user_id = "ib-xPv06AjuYOE4BI0"
    print(f"--- Recommendations for User Index: {target_user_idx} ---")


    history = als_data.filter(col("userIndex") == target_user_idx).join(item_lookup, "itemIndex")

    reco_10 = (clean_recommendations
        .filter(col("userIndex") == target_user_idx)
        .join(item_lookup, "itemIndex")
        .orderBy(col("score").desc())
        .limit(10))

    # Show History
    print("\n[ History ]")
    history = history.join(enriched_content_df,"item_id", how="left").select("item_id", "title", "original_language", "Genres")
    history.show(10, truncate=False)

    # Show Clean Recs
    print("\n[ ALS Suggestions ]")
    reco_10 = reco_10.join(enriched_content_df,"item_id", how = "left").select("item_id", "title", "original_language", "Genres")
    reco_10.show(10, truncate=False)

    print(f"--- LightFM Recommendations for User Index: {target_user_idx} ---")
    # Get the Pandas DataFrame from your function
    lightfm_pandas_df = recommend_for_user(target_user_idx, lfm_model, interactions, item_features)

    # Convert it to Spark and print using the exact same formatting!
    lightfm_spark_df = spark.createDataFrame(lightfm_pandas_df)
    lightfm_spark_df.select("item_id", "title", "original_language", "Genres").show(10, truncate=False)
else:
    print("User ID not found.", result)



26/03/18 07:33:13 WARN DAGScheduler: Broadcasting large task binary with size 77.9 MiB
26/03/18 07:33:19 WARN DAGScheduler: Broadcasting large task binary with size 77.8 MiB
26/03/18 07:33:32 WARN DAGScheduler: Broadcasting large task binary with size 89.6 MiB


User ID not found. []


In [25]:
watch_history_df.filter(F.col("userId") == "ib-xPv06AjuYOE4BI0").show(20, truncate=False)

+------------------+-----------------------------------+-------------------+----------+
|userId            |item_id                            |total_play_time_sec|day       |
+------------------+-----------------------------------+-------------------+----------+
|ib-xPv06AjuYOE4BI0|LIONSGATEPLAY_MOVIE_12STRONG0Y2018M|2.0                |2026-03-12|
+------------------+-----------------------------------+-------------------+----------+



26/03/18 06:48:58 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 64 for reason Executor for container container_1764236692086_5240_01_000069 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 06:48:58 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 67 for reason Executor for container container_1764236692086_5240_01_000072 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 06:49:01 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 65 for reason Executor for container container_1764236692086_5240_01_000070 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/18 06:49:01 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 66 for reason Executor for container container_1764236692086_5

In [26]:
als_input_base.filter(F.col("userId") == "ib-xPv06AjuYOE4BI0").show(10, truncate=False)

+------+-------+-----------------------+----------------------+
|userId|item_id|total_playtime_combined|distinct_content_count|
+------+-------+-----------------------+----------------------+
+------+-------+-----------------------+----------------------+



In [22]:
clean_recommendations.show(10, truncate=False)

+---------+---------+----------+
|userIndex|itemIndex|score     |
+---------+---------+----------+
|30       |9916     |0.15185788|
|57       |578      |0.26766372|
|116      |844      |0.6052091 |
|137      |6694     |0.46229008|
|142      |27084    |0.4444842 |
|177      |21123    |0.44679546|
|258      |19816    |0.29309514|
|270      |13212    |0.27709183|
|337      |26813    |0.273558  |
|386      |135      |0.0       |
+---------+---------+----------+
only showing top 10 rows

